##Load and inspect Bronze bed transfer events

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE = f"{CATALOG}.bronze_schema"
SILVER = f"{CATALOG}.silver_schema"
transfers_bronze = spark.table(f"{BRONZE}.bronze_bed_transfer_events")
display(transfers_bronze.limit(10))
print("Total transfer rows:", transfers_bronze.count())

transfer_id,encounter_id,patient_id,from_bed_id,to_bed_id,transfer_ts,transfer_reason,department_id,transfer_status,recorded_ts,_rescued_data,source_file,ingestion_timestamp
TRANSFE080000001,ENCOUNT080000030,PATIEN0800086,FROM_BE080000001,TO_BED080000001,2026-07-01T04:40:00Z,ISOLATION,DEPART0800104,INACTIVE,2026-07-01T05:00:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000002,ENCOUNT080000059,PATIEN0800117,FROM_BE080000002,TO_BED080000002,2026-07-01T04:47:00Z,CAPACITY,DEPART0800114,ACTIVE,2026-07-01T05:07:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000003,ENCOUNT080000088,PATIEN0800069,FROM_BE080000003,TO_BED080000003,2026-07-01T04:54:00Z,ISOLATION,DEPART0800057,PENDING,2026-07-01T05:14:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000004,ENCOUNT080000117,PATIEN0800159,FROM_BE080000004,TO_BED080000004,2026-07-01T05:01:00Z,CAPACITY,DEPART0800169,INACTIVE,2026-07-01T05:21:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000005,ENCOUNT080000146,PATIEN0800038,FROM_BE080000005,TO_BED080000005,2026-07-01T05:08:00Z,CAPACITY,DEPART0800019,ACTIVE,2026-07-01T05:28:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000006,ENCOUNT080000175,PATIEN0800161,FROM_BE080000006,TO_BED080000006,2026-07-01T05:15:00Z,CAPACITY,DEPART0800167,ACTIVE,2026-07-01T05:35:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000007,ENCOUNT080000204,PATIEN0800045,FROM_BE080000007,TO_BED080000007,2026-07-01T05:22:00Z,CLINICAL,DEPART0800188,ACTIVE,2026-07-01T05:42:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000008,ENCOUNT080000233,PATIEN0800140,FROM_BE080000008,TO_BED080000008,2026-07-01T05:29:00Z,ISOLATION,DEPART0800127,PENDING,2026-07-01T05:49:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000009,ENCOUNT080000262,PATIEN0800094,FROM_BE080000009,TO_BED080000009,2026-07-01T05:36:00Z,CLINICAL,DEPART0800051,ACTIVE,2026-07-01T05:56:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
TRANSFE080000010,ENCOUNT080000291,PATIEN0800006,FROM_BE080000010,TO_BED080000010,2026-07-01T05:43:00Z,CLINICAL,DEPART0800056,ACTIVE,2026-07-01T06:03:00Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z


Total transfer rows: 392


##Clean bed transfer event columns

In [0]:
transfers_checked = (
    transfers_bronze
    .withColumn("transfer_id", F.trim("transfer_id"))
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("from_bed_id", F.trim("from_bed_id"))
    .withColumn("to_bed_id", F.trim("to_bed_id"))
    .withColumn("transfer_ts", F.to_timestamp("transfer_ts"))
    .withColumn("transfer_reason", F.upper(F.trim("transfer_reason")))
    .withColumn("department_id", F.trim("department_id"))
    .withColumn("transfer_status", F.upper(F.trim("transfer_status")))
    .withColumn("recorded_ts", F.to_timestamp("recorded_ts"))
)

##Check mandatory fields and transfer chronology

In [0]:
transfers_checked = transfers_checked.withColumn(
    "dq_reason",
    F.concat_ws(", ",
        F.when(F.col("transfer_id").isNull() | (F.col("transfer_id") == ""), "MISSING_TRANSFER_ID"),
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "MISSING_ENCOUNTER_ID"),
        F.when(F.col("transfer_ts").isNull(), "INVALID_TRANSFER_TIMESTAMP"),
        F.when(F.col("recorded_ts").isNotNull() & F.col("transfer_ts").isNotNull() & (F.col("recorded_ts") < F.col("transfer_ts")), "RECORDED_BEFORE_TRANSFER")
    )
)

transfers_quarantine = transfers_checked.filter(F.col("dq_reason") != "")
transfers_valid = transfers_checked.filter(F.col("dq_reason") == "")

##Validate encounter relationship

In [0]:
encounter_keys = spark.table(f"{SILVER}.silver_patient_encounters").select("encounter_id").distinct()

transfers_valid = transfers_valid.join(
    encounter_keys.withColumn("encounter_found", F.lit(1)), "encounter_id", "left"
).withColumn(
    "dq_reason",
    F.when(F.col("encounter_found").isNull(), "UNKNOWN_ENCOUNTER_ID").otherwise("")
)

transfers_quarantine = transfers_quarantine.unionByName(
    transfers_valid.filter(F.col("dq_reason") != ""), allowMissingColumns=True
)
transfers_valid = transfers_valid.filter(F.col("dq_reason") == "")

##Deduplicate Bed Transfer Events

In [0]:
transfer_window = Window.partitionBy("transfer_id").orderBy(
    F.col("ingestion_timestamp").desc_nulls_last(),
    F.col("source_file").asc_nulls_last()
)

transfers_ranked = transfers_valid.withColumn("row_num", F.row_number().over(transfer_window))
transfers_duplicate_audit = transfers_ranked.filter(F.col("row_num") > 1)
transfers_silver = transfers_ranked.filter(F.col("row_num") == 1).drop("row_num", "dq_reason", "encounter_found")

##Save Transfer Silver, Quarantine and Duplicate Audit

In [0]:
(transfers_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_bed_transfer_events"))

(transfers_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_transfer_quarantine"))

(transfers_duplicate_audit.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_transfer_duplicate_audit"))

##Validate saved Transfer tables

In [0]:
for table_name in ["silver_bed_transfer_events", "silver_transfer_quarantine", "silver_transfer_duplicate_audit"]:
    print(table_name, spark.table(f"{SILVER}.{table_name}").count())

display(spark.table(f"{SILVER}.silver_bed_transfer_events").limit(10))

silver_bed_transfer_events 392
silver_transfer_quarantine 0
silver_transfer_duplicate_audit 0


encounter_id,transfer_id,patient_id,from_bed_id,to_bed_id,transfer_ts,transfer_reason,department_id,transfer_status,recorded_ts,_rescued_data,source_file,ingestion_timestamp
ENCOUNT080000030,TRANSFE080000001,PATIEN0800086,FROM_BE080000001,TO_BED080000001,2026-07-01T04:40:00.000Z,ISOLATION,DEPART0800104,INACTIVE,2026-07-01T05:00:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000059,TRANSFE080000002,PATIEN0800117,FROM_BE080000002,TO_BED080000002,2026-07-01T04:47:00.000Z,CAPACITY,DEPART0800114,ACTIVE,2026-07-01T05:07:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000088,TRANSFE080000003,PATIEN0800069,FROM_BE080000003,TO_BED080000003,2026-07-01T04:54:00.000Z,ISOLATION,DEPART0800057,PENDING,2026-07-01T05:14:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000117,TRANSFE080000004,PATIEN0800159,FROM_BE080000004,TO_BED080000004,2026-07-01T05:01:00.000Z,CAPACITY,DEPART0800169,INACTIVE,2026-07-01T05:21:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000146,TRANSFE080000005,PATIEN0800038,FROM_BE080000005,TO_BED080000005,2026-07-01T05:08:00.000Z,CAPACITY,DEPART0800019,ACTIVE,2026-07-01T05:28:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000175,TRANSFE080000006,PATIEN0800161,FROM_BE080000006,TO_BED080000006,2026-07-01T05:15:00.000Z,CAPACITY,DEPART0800167,ACTIVE,2026-07-01T05:35:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000204,TRANSFE080000007,PATIEN0800045,FROM_BE080000007,TO_BED080000007,2026-07-01T05:22:00.000Z,CLINICAL,DEPART0800188,ACTIVE,2026-07-01T05:42:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000233,TRANSFE080000008,PATIEN0800140,FROM_BE080000008,TO_BED080000008,2026-07-01T05:29:00.000Z,ISOLATION,DEPART0800127,PENDING,2026-07-01T05:49:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000262,TRANSFE080000009,PATIEN0800094,FROM_BE080000009,TO_BED080000009,2026-07-01T05:36:00.000Z,CLINICAL,DEPART0800051,ACTIVE,2026-07-01T05:56:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
ENCOUNT080000291,TRANSFE080000010,PATIEN0800006,FROM_BE080000010,TO_BED080000010,2026-07-01T05:43:00.000Z,CLINICAL,DEPART0800056,ACTIVE,2026-07-01T06:03:00.000Z,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/event_data/bed_transfer_events.csv,2026-11-04T06:17:17.892Z
